# Day 1 — Kafka lab (Jupyter + CMD)

**Before this notebook:** complete the Setup section in [commands.md](commands.md) — the JDK, the AWS CLI, the Kafka download, and one run of `start-lab.bat`.

### What you will be able to do by the end of today

| Skill | Why it matters for the rest of the week |
|-------|------------------------------------------|
| Connect to MSK from outside its VPC, over TLS with SCRAM | Every command on Days 2 to 5 depends on this working |
| Read a topic's partitions, leaders and ISR | This is how you check whether data is healthy |
| Produce and consume messages from the command line | Your test instrument for proving a pipeline works |
| Read a consumer group's offsets and lag | The single most useful diagnostic in Kafka support |

### How to work through the notebook

Run the cells **top to bottom** with Shift+Enter. Read the **Concept** cell before each command — it explains *why* you are running it, which is what you will actually need when there is no notebook in front of you.

Each code cell runs the same CMD commands as [commands.md](commands.md), so anything you learn here works identically in a plain Command Prompt.

**About the `log4j WARN` lines:** every Kafka command on Windows prints one or two. They mean the CLI has no logging configuration file. They are not errors, and they do not mean your command failed.

Full theory: [notes.md](notes.md).


## B — Load lab session

**Every Jupyter cell starts a brand-new CMD process.** Environment variables do not survive from one cell to the next, which is why every single cell below begins with the same line:

`call ..\scripts\jupyter-lab-session.bat`

That loads `%USERPROFILE%\set-kafka-lab.bat`, which sets the values every command needs:

| Variable | What it holds |
|----------|---------------|
| `%TOPIC%` | Your topic, `orders-userN` |
| `%GROUP%` | Your consumer group, `cg-userN-support` |
| `%BOOTSTRAP%` | The broker address your client connects to first |
| `%CLIENT%` | The properties file holding your SCRAM username and password |

If you forget that line, the command runs with empty variables and fails in a confusing way — usually a hang, or a complaint about a missing argument. When a cell behaves strangely, this is the first thing to check.

**What you should see:** your own seat ids (`user1`, `user2` and so on) and a bootstrap string containing **`-public`** in the hostname, ending in port **`:9196`**.


### Concept: Your seat identity

You have a seat number, `userN`, and it is used consistently in five places:

| Where | Value |
|-------|-------|
| AWS Console login | `userN` |
| SCRAM username in your client properties file | `userN` |
| Your topic | `orders-userN` |
| Your consumer group | `cg-userN-support` |
| Your ACL practice topic (Day 4) | `acl-lab-userN` |

`set-kafka-lab.bat` puts these into environment variables, so the commands throughout the week refer to `%TOPIC%` and `%GROUP%` rather than hard-coded names. That means the same command text works for every seat.

### Why staying on your own ids matters

Everyone shares one MSK cluster. Working on somebody else's topic or consumer group is not merely impolite — it actively breaks their lab. Resetting another seat's offsets, or consuming from their group, changes the committed position they are in the middle of measuring.

This is not just a classroom rule. In production, a shared Kafka cluster carries many teams' topics, and the same discipline applies: confirm the topic and group names belong to the system you were asked to work on, before you run anything that changes state.


In [1]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo LOGIN=%LOGIN% TOPIC=%TOPIC% GROUP=%GROUP%
echo BOOTSTRAP=%BOOTSTRAP%
echo JAVA_HOME=%JAVA_HOME%

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>echo LOGIN=%LOGIN% TOPIC=%TOPIC% GROUP=%GROUP%
LOGIN=user15 TOPIC=orders-user15 GROUP=cg-user15-support

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>echo BOOTSTRAP=%BOOTSTRAP%
BOOTSTRAP=b-1-public.mskkafkaclass.qau5zr.c4.kafka.ap-south-1.amazonaws.com:9196

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>echo JAVA_HOME=%JAVA_HOME%
JAVA_HOME=C:\PROGRA~1\Java\jdk-21

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

## 1 — Connect (AWS + Kafka)

Before you can troubleshoot anything, you must prove that **two separate paths** work from your Windows lab PC.

| Path | What it is | How you test it | If it fails |
|------|-----------|-----------------|-------------|
| **Control plane** | The AWS APIs that manage the cluster (`aws kafka ...`) | `aws sts get-caller-identity`, `describe-cluster` | You may be in the wrong AWS account — Kafka data commands can still work |
| **Data plane** | Kafka itself: producing and consuming (`kafka-*.bat`) | `kafka-topics --list` over SCRAM on port 9196 | Check the bootstrap host and port — this is what really matters for produce/consume |

The two paths use **different credentials**. Your AWS CLI login can fail while your Kafka SCRAM login still works, and the other way round. Keeping them separate in your mind avoids a very common support mistake: blaming Kafka when only the AWS API call was denied.


### Concept: two different ways to talk to MSK

This is the most useful idea in section 1, and it explains an error that confuses almost everyone on their first day.

An MSK cluster is reached through **two entirely separate channels**:

| Channel | What it does | How you authenticate | Commands |
|---------|--------------|----------------------|----------|
| **AWS control plane** | Asks AWS *about* the cluster — its state, its broker addresses, its configuration | AWS IAM credentials, from `aws configure` | `aws kafka describe-cluster`, `aws kafka get-bootstrap-brokers` |
| **Kafka data plane** | Talks *to* the brokers — topics, messages, offsets | SCRAM username and password, from `%CLIENT%` | `kafka-topics.bat`, `kafka-console-producer`, and everything else |

**They are independent.** Different credentials, different network path, different ports.

### Why that matters right now

If `aws kafka describe-cluster` returns **`AccessDenied`**, it means your AWS CLI credentials are for a different account, or lack the MSK permission. It says **nothing at all** about whether Kafka works.

Your Kafka commands can still succeed, because they use SCRAM and the bootstrap address that `set-kafka-lab.bat` already gave you. So if you hit `AccessDenied` here, note it and carry on to section 1.4 — the Kafka path is what today is really about.

### The three commands in this section

| Command | The question it answers |
|---------|-------------------------|
| `aws sts get-caller-identity` | Which AWS account and user am I signed in as? |
| `aws kafka describe-cluster` | Is the cluster **ACTIVE**, and how many brokers does it have? |
| `aws kafka get-bootstrap-brokers` | Which addresses should my client connect to? |


In [2]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws sts get-caller-identity

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>aws sts get-caller-identity
{
    "UserId": "AIDAV7A532FCXXK2WI57D",
    "Account": "410232017221",
    "Arn": "arn:aws:iam::410232017221:user/u1"
}

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Reading — AWS identity

`aws sts get-caller-identity` returns three fields. The one to look at is **`Account`**.

| What you see | What it means | What to do |
|--------------|---------------|------------|
| `Account` matches the lab account | Your AWS CLI can query MSK | Continue — both channels work |
| A different account number | Your CLI is configured for somewhere else | Note it. The Kafka commands still work over SCRAM. |
| `Unable to locate credentials` | No AWS CLI configuration on this machine at all | Run `aws configure`, or continue with just the Kafka commands |
| `describe-cluster` returns `AccessDenied` | Authentication worked; this user lacks the MSK permission | Skip to section 1.4 and use the Kafka path |

**The distinction worth internalising:** `AccessDenied` and an authentication failure are different results. `AccessDenied` means AWS knows exactly who you are and has decided you may not do that particular thing. That is a permissions question, not a credentials question — and it is the same distinction you will meet again on Day 4 as authentication versus authorization.


## 1.2 — MSK cluster state

**What you should see:** JSON from AWS containing `"State": "ACTIVE"` and a broker count of **three**. That means the cluster is running and ready for clients.

### The cluster states you might meet

| State | Meaning | What you would do about it |
|-------|---------|----------------------------|
| **ACTIVE** | Normal operation | Nothing — carry on |
| **UPDATING** | A configuration change or version upgrade is in progress. Brokers restart one at a time. | Wait. Clients may briefly see leader changes. |
| **HEALING** | AWS is replacing a failed broker | Expect under-replicated partitions until it finishes. Monitor rather than change anything. |
| **MAINTENANCE** | AWS-initiated patching | Wait |
| **CREATING** / **DELETING** | The cluster is being built or removed | Not usable |

Recognising **HEALING** is genuinely useful later in the week: it explains under-replicated partitions and brief produce failures without anyone having made a mistake, and the correct response is to wait and watch rather than start changing configuration.

**If you get `AccessDenied`:** your AWS CLI user is not in the lab account. Note it and continue — the Kafka commands below use SCRAM and the bootstrap address already loaded in section B, so they are unaffected.


In [3]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION%

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>aws kafka describe-cluster --cluster-arn %CLUSTER_ARN% --region %REGION%



aws: [ERROR]: An error occurred (AccessDeniedException) when calling the DescribeCluster operation: User: arn:aws:iam::410232017221:user/u1 is not authorized to perform: kafka:DescribeCluster on resource: arn:aws:kafka:ap-south-1:891377046325:cluster/msk-kafka-class/ad474b96-f594-495b-82cd-ad95d7c2c71c-4 because no resource-based policy allows the kafka:DescribeCluster action



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

## 1.3 — Bootstrap brokers

### What a "bootstrap server" actually is

It is only a **starting point**, not the server your client keeps using.

Your client connects to the bootstrap address, asks *"who is in this cluster, and which broker leads which partition?"*, and receives the full list back. From then on it talks directly to whichever broker holds the data it needs. That is why a single bootstrap address is enough to reach a three-broker cluster.

### Why MSK gives you several different strings

The same cluster hands out different addresses depending on where your client sits and how it authenticates:

| Field name contains | Port | Who uses it |
|---------------------|------|-------------|
| `PublicSaslScram` | **9196** | Clients **outside** the VPC, using a username and password — **this is you** |
| `PublicSaslIam` | 9198 | Clients outside the VPC, using AWS IAM (Day 4) |
| `SaslScram` (no "Public") | 9096 | Clients **inside** the VPC |
| `SaslIam` (no "Public") | 9098 | Clients inside the VPC, using IAM |

**What you should see:** a field like `"BootstrapBrokerStringPublicSaslScram"` — the exact name varies slightly by MSK version — holding a **`-public`** hostname on port **`:9196`**.

**Getting this wrong is the most common Kafka connection problem there is.** From outside the VPC, the private `:9096` address does not fail with a helpful message. It simply hangs and then times out, which looks exactly like a broken cluster. You will diagnose that exact symptom on Day 5.

**If you get `AccessDenied`:** use the `BOOTSTRAP` value already printed in section B. It came from `set-kafka-lab.bat` and is the same string.


In [4]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
aws kafka get-bootstrap-brokers --cluster-arn %CLUSTER_ARN% --region %REGION%

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>aws kafka get-bootstrap-brokers --cluster-arn %CLUSTER_ARN% --region %REGION%



aws: [ERROR]: An error occurred (AccessDeniedException) when calling the GetBootstrapBrokers operation: User: arn:aws:iam::410232017221:user/u1 is not authorized to perform: kafka:GetBootstrapBrokers on resource: arn:aws:kafka:ap-south-1:891377046325:cluster/msk-kafka-class/ad474b96-f594-495b-82cd-ad95d7c2c71c-4 because no resource-based policy allows the kafka:GetBootstrapBrokers action



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Concept: Topics & partitions

A **topic** is a named stream (your `orders-userN`). It is split into **partitions** (usually 3 in this lab).

- **Leader** — broker handling reads/writes for that partition
- **Replicas / ISR** — copies; healthy topic has full ISR (all 3 brokers in sync)
- **`--list`** — first proof you can reach Kafka with your SCRAM login

**What you should see:** three partitions, replication factor **3**, and **Isr** matching **Replicas** on each partition. Yellow **log4j WARN** lines on Windows are normal — you can ignore them.


## 1.4 — List topics (first Kafka call)

This is your first command that talks to the **brokers** rather than to the AWS API, and it quietly proves four things at once:

| It proves | Because |
|-----------|---------|
| Network connectivity | Your machine reached the broker on port 9196 |
| TLS works | The encrypted connection was negotiated successfully |
| SCRAM authentication works | The broker accepted the username and password in `%CLIENT%` |
| You have permission to list topics | Authorization allowed the operation |

That is why `--list` is the first thing to run whenever someone reports a connection problem. One command, four answers.

**What you should see:** a list of topics including your own `orders-userN`, returned within a few seconds. You will also see other seats' topics and some AWS internal ones — that is normal on a shared cluster.

**If it hangs and then times out**, work through these in order:

1. Is `%BOOTSTRAP%` a **`-public`** hostname on port **`:9196`**? The private `9096` endpoint is unreachable from here.
2. Did the cell start with `call ..\scripts\jupyter-lab-session.bat`? Without it, `%BOOTSTRAP%` is empty.
3. Does `%CLIENT%` point at a file that exists, containing your SCRAM username and password?


In [5]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


acl-lab-user15
orders-demo
orders-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

## 2 — Describe your topic

`--list` only proves that a topic **exists**. To troubleshoot you need its **shape and health**, and that is what `--describe` shows:

- **Partition count** — how the topic is split so work can run in parallel
- **Replication factor (RF)** — how many copies of each partition exist
- **Leader** — the broker that serves reads and writes for each partition
- **Replicas / ISR** — all copies, and the ones that are currently **in sync**

Read the next output as your topic's "health card". You will run this same command every day this week, so get comfortable with it now.


### Concept: two describe commands, two different answers

These look interchangeable and are not. Running the wrong one and seeing nothing useful leads people to conclude a setting does not exist.

| Command | What it shows |
|---------|---------------|
| `kafka-topics --describe` | Structure and health — partition count, replication factor, which broker leads each partition, and which replicas are in sync. Also shows the **effective** configuration on the topic line. |
| `kafka-configs --describe` | Only the **overrides** set specifically on this topic. Frequently empty. |

**An empty `kafka-configs --describe` is a normal, healthy result.** It means nobody has overridden anything, so the topic is inheriting the broker defaults — `min.insync.replicas=2` and the default retention. It does **not** mean the topic has no configuration.

This distinction returns on Day 4, when the question *"what is the retention on this topic?"* becomes a real one. The answer is: check the override first, and fall back to the broker default if there is no override.


In [10]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo TOPIC=%TOPIC%
kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>echo TOPIC=%TOPIC%
TOPIC=orders-user15

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-topics.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --topic %TOPIC% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Topic: orders-user15	TopicId: u5fx7o-2RfW2LvjgNRKbPw	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,message.format.version=3.0-IV1,unclean.leader.election.enable=true
	Topic: orders-user15	Partition: 0	Leader: 2	Replicas: 2,1,3	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 1	Leader: 1	Replicas: 1,3,2	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A
	Topic: orders-user15	Partition: 2	Leader: 3	Replicas: 3,2,1	Isr: 2,3,1	Elr: N/A	LastKnownElr: N/A

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

## 3 — Topic configuration

**What you should see:** most likely an **empty** result, or a single line with no values after it.

That is the correct outcome and not a sign that anything is missing. `kafka-configs --describe` lists only settings explicitly overridden on this topic. Your topic was created without overrides, so it inherits the broker defaults.

| If you see | It means |
|------------|----------|
| Nothing after the topic name | No overrides. Broker defaults apply. |
| `retention.ms=...` or similar | Somebody set that value on this topic specifically, and it wins over the broker default |

**Why bother running it at all?** Because when a topic behaves differently from its neighbours — filling disk faster, or deleting data sooner than expected — an override is very often the reason, and this is the only command that reveals one.


In [11]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-configs.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --entity-type topics --entity-name %TOPIC% --describe


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


Dynamic configs for topic orders-user15 are:

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Concept: Consumer groups

A **consumer group** is a named set of consumers that share the work of reading a topic and share a single record of how far they have got.

### The two things a group does

**1. It divides the work.** Each partition is assigned to exactly one member of the group. Three partitions and three members means one partition each. Three partitions and one member means that member reads all three. Adding a fourth member to a three-partition topic gains you nothing — there is no fourth partition to give it.

**2. It remembers the position.** The group commits an **offset** per partition, recording how far it has read. That is what lets a consumer restart and carry on from where it stopped, instead of re-reading everything.

### Why this is the most important concept in Kafka support

Almost every "data is missing" or "data is late" ticket comes down to one of these:

| The situation | What you see in `--describe` |
|---------------|------------------------------|
| Nothing is reading | No members, and `LAG` growing |
| Reading, but too slowly | Members present, `LAG` growing |
| Caught up and healthy | Members present, `LAG` at 0 |
| Position is wrong | `CURRENT-OFFSET` is not where it should be |

### Two normal things that look like errors

- **`does not exist` before your first consume.** A group is created when a consumer first joins it, not in advance. This message on Day 1 is expected.
- **`amazon.msk.canary.group.*` in `--list`.** These are AWS's own internal health-check groups. Ignore them.


## 4 — Consumer groups

A **consumer group** tracks how far a set of consumers has read on each partition — its **offsets**. Two commands help you here:

- **`--list`** — shows every group on the cluster. You will also see AWS internal `amazon.msk.canary.group.*` groups (ignore those) and the `cg-userN-support` groups belonging to other seats.
- **`--describe --group %GROUP%`** — shows your group's per-partition offsets and lag.

**Why the next describe shows an error:** your group `cg-userN-support` does **not exist yet**, because you have not consumed any messages with it. The line `Error: Consumer group '...' does not exist.` is the **expected, normal** result before section 5. Kafka creates the group the first time a consumer in it reads and commits an offset.


In [12]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --list


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


amazon.msk.canary.group.broker-1
cg-user7-support
cg-user6-support
cg-user8-support
cg-user2-support
cg-user10-support
cg-user1-support
cg-user3-support
cg-user12-support
cg-user4-support
amazon.msk.canary.group.broker-3
cg-user5-support
amazon.msk.canary.group.broker-2

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

In [13]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
echo GROUP=%GROUP%
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>echo GROUP=%GROUP%
GROUP=cg-user15-support

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



Error: Consumer group 'cg-user15-support' does not exist.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Concept: Message lifecycle & offsets

Every message on a partition gets a sequential **offset** (0, 1, 2, …).

| Stage | What happens |
|-------|----------------|
| **Produce** | Client sends to partition **leader**; followers replicate (RF=3) |
| **Commit** | Consumer group saves **CURRENT-OFFSET** after processing |
| **LAG** | LOG-END-OFFSET minus CURRENT-OFFSET — backlog waiting |

**`--from-beginning` vs default:** first consume on a new group reads from **earliest** retained offset; later runs continue from the **committed** bookmark unless you reset.

### Concept: Produce → consume happy path

1. **Producer** writes to the partition leader (keys like `order-1001` hash to a partition)
2. **Consumer** in `%GROUP%` reads and advances offset
3. **`--describe`** shows LAG → **0** when caught up

In production, producer and consumer run in **separate processes**. Here we produce two lines, then consume with `--from-beginning` to see them replayed.


## 5a — Produce messages

You will send **three** messages. Notice that `order-1001` is sent **twice**, plus one `order-1002`:

`order-1001` → `order-1002` → `order-1001` again

Each line is one message written to a partition of `%TOPIC%`. Messages with the **same key** always go to the **same partition**, so both `order-1001` lines land on the same partition.

**What you should see:** only `log4j WARN` lines, then the prompt returns with no `ERROR` and no timeout. That means all three messages were **published successfully**. The producer does not echo the message body, so a quiet prompt here is the sign of success.


In [14]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat order-1001
call ..\scripts\produce.bat order-1002
call ..\scripts\produce.bat order-1001


Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\produce.bat order-1001


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\produce.bat order-1002


log4j:WARN No appenders could be found for logger (kafka.utils.Log4jControllerRegistration$).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Reading — partition by key

Now think about **where** those messages landed. Kafka chooses a partition from the message **key**:

- Same key → **same partition**, every time. Your two `order-1001` messages share one partition.
- Different key → may hash to a **different** partition. `order-1002` can land elsewhere.

**Why this matters for support:** uneven lag on a single partition is often a **hot partition** — many messages sharing one key — not a broken broker. Always check lag **per partition** before you escalate to "the cluster is overloaded" or ask for more brokers.


## 5b — Consume messages

This runs the same thing as `consume.bat --from-beginning`, with a message limit and a timeout so the cell finishes on its own.

### Why the flags are there

| Flag | Purpose |
|------|---------|
| `--from-beginning` | Start at the oldest retained message — but **only if this group has no saved position yet**. If it already has committed offsets, Kafka uses those instead. |
| `--max-messages` | Stop after N messages, so the cell cannot run forever |
| `--timeout-ms` | Give up after this long if nothing more arrives |

On a real server you would leave the consumer running and stop it with Ctrl+C when you had seen enough. A notebook cell cannot be interrupted that way, hence the limits.

**What you should see:** your message lines — `order-1001`, `order-1002` — followed by `Processed a total of N messages`.

**This cell can take around 20 seconds.** That is the consumer waiting out its poll timeout to be sure no more messages are coming. Slowness here is expected, not a problem.

### What just happened underneath

Running this command created your consumer group, joined it, was assigned the topic's partitions, read the available messages, and committed an offset for each partition. That commit is why section 5d will behave differently.


In [15]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\consume.bat --from-beginning --max-messages 20 --timeout-ms 20000

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\consume.bat --from-beginning --max-messages 20 --timeout-ms 20000


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.consumer.ConsumerConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


order-1001
order-1002


Processed a total of 2 messages



c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

## 5c — Group describe (after consume)

Now that the group exists, `--describe` returns the table you will use constantly for the rest of the week.

### The three columns that matter

| Column | Meaning |
|--------|---------|
| **`CURRENT-OFFSET`** | How far this group has committed — the bookmark |
| **`LOG-END-OFFSET`** | How far the topic has been written to |
| **`LAG`** | `LOG-END-OFFSET` minus `CURRENT-OFFSET` — messages written but not yet processed |

**What you should see:**

- Your group name, `cg-userN-support`
- **`LAG` of 0** on the partitions that received your test messages — the group is caught up
- Possibly `0` in all three columns for one partition. That is normal: your messages went to the partitions their keys selected, and one partition may not have received any.
- **No active members**, because the consumer in 5b has already exited. An empty `CONSUMER-ID` here is expected.

### The reading habit to build now

Read `LAG` **per partition**, never just the total. A group can be perfectly healthy on two partitions and badly behind on a third — usually because one partition is far busier than the others, or one consumer instance is struggling. The total hides exactly the thing you need to see, and this is the difference between "the consumer is slow" and "one partition is hot".


## 5d — Consume without `--from-beginning` (committed offset)

Run this **after** section 5b, because the whole point is what changed in between.

Without `--from-beginning`, the consumer starts from the group's **committed offset** — the bookmark that 5b saved. So it reads only messages that arrived *after* 5b finished.

**What you should see:** often **nothing at all**, or only messages you produced after 5b. Then `Processed a total of 0 messages`.

**An empty result here is a success.** It is the proof that offset committing works: the group remembered its position, so it did not re-read messages it had already handled.

### The comparison to hold onto

| Run | Where it starts | What you see |
|-----|-----------------|--------------|
| 5b, with `--from-beginning`, no saved offset yet | The oldest retained message | All your test messages |
| 5d, no flag, offset already saved | The committed offset | Only newer messages, often none |

This is the mechanism behind every offset conversation for the rest of the week. On Day 4 you will deliberately move that bookmark backwards to replay messages, and it works precisely because of what you just observed here.

**One subtlety worth knowing:** `--from-beginning` is not a command to "always start at the beginning". It only applies when the group has **no** committed offset. Once a position is saved, Kafka uses the saved position and the flag has no effect — which is why re-running 5b later does not replay everything again.


In [ ]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
call ..\scripts\produce.bat order-1003
call ..\scripts\consume.bat --max-messages 5 --timeout-ms 25000


### Reading — offset contrast

| Run | Flag | What you read |
|-----|------|---------------|
| **5b — first consume** | `--from-beginning` | Reads from the **oldest** message still kept on the topic, then saves the group's position (commits offsets) |
| **5d — second consume** | default (no flag) | Reads only messages **after** the saved position — here that should be just `order-1003` |

This contrast is the whole point of offsets: once a group commits its position, it does **not** re-read old messages unless you explicitly reset it (you will do that on Day 4).

**If 5d shows 0 messages:** wait the full timeout (about 25 seconds) — the consumer group may still be joining. Then produce `order-1003` again and re-run 5d. On a clean first run right after 5b, you should see **exactly one** new line.

**On real tickets:** a reported "missing message" is very often **already consumed past** — the group's offset moved forward — not actually deleted from Kafka. Always check the committed offset before you assume data loss.


In [16]:
%%cmd
call ..\scripts\jupyter-lab-session.bat
kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe

Microsoft Windows [Version 10.0.26200.9278]
(c) Microsoft Corporation. All rights reserved.

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>call ..\scripts\jupyter-lab-session.bat

c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>kafka-consumer-groups.bat --bootstrap-server %BOOTSTRAP% --command-config %CLIENT% --group %GROUP% --describe


log4j:WARN No appenders could be found for logger (org.apache.kafka.clients.admin.AdminClientConfig).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.

Consumer group 'cg-user15-support' has no active members.



GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
cg-user15-support orders-user15   0          1               1               0               -               -               -
cg-user15-support orders-user15   1          1               1               0               -               -               -
cg-user15-support orders-user15   2          0               0               0               -               -               -
c:\25-Trainings\2-Confirmed\20-08-26-kafka\GH\day-01>

### Reading — happy path complete

| Check | Healthy sign |
|-------|-------------|
| `--list` | `%TOPIC%` present |
| `--describe` topic | RF=3, ISR = all replicas |
| produce | No ERROR / timeout |
| consume 5b | `order-1001`, `order-1002` printed |
| consume 5d | Only new line (`order-1003`) |
| group `--describe` | CURRENT-OFFSET > 0; **LAG 0** |

**Topic card worksheet (assignment):** fill partition count, RF, leader ids, ISR healthy?, one-sentence message flow.

**Next (Day 2):** when LAG > 0 or logs show timeouts, diagnose before changing anything.


## Assignment

Using the output from the cells above, fill in [samples/assignment-topic-card.md](samples/assignment-topic-card.md):

- Partition count and replication factor (from the §2 `--describe`)
- The leader broker id for each partition
- Whether **ISR equals Replicas** (is the topic healthy?)
- One sentence describing the message flow from producer to your consumer group

This "topic card" is your baseline. On later days, when something looks wrong, you compare the live topic against this healthy snapshot.
